In [1]:
# make top level dir available
import sys
sys.path.append("../../")

import numpy as np
import functions as myfunc
import meshtools as mt
import matplotlib.pyplot as plt
import scipy as sci

from typing import Optional

ModuleNotFoundError: No module named 'numpy'

## Load Data

In [ ]:
# custom values
EDGE_LENGTH = 1e-3  # in m

# given paramters
OUTER_RADIUS = 0.06     # in m
INNER_RADIUS = 0.02     # in m
MATERIAL_WIDTH = 0.004  # in m
HEIGHT = 0.1            # in m

CONDUCTIVITY_1 = 1      # in S/m
CONDUCTIVITY_2 = 10     # in S/m
CONDUCTIVITY_3 = 0.05   # in S/m
CONDUCTIVITY_4 = 20     # in S/m

CONNECTOR_ANGLE = 10 * np.pi/180   # in rad

CONNECTOR_1_VOLTAGE = 1                     # V
CONNECTOR_2_VOLTAGE = -CONNECTOR_1_VOLTAGE  # V

In [ ]:
nodes, elements, boundary_edge, boundary_indices, boundary_elements, outer_curve, inner_curve = mt.LoadTriMesh("Netz_SS25.npz", show=True)

In [ ]:
nodes.shape

In [ ]:

# points on the outer circle
position_9  = myfunc.assemble_point(9,  OUTER_RADIUS, 7/4*np.pi - CONNECTOR_ANGLE/2)
position_10 = myfunc.assemble_point(10, OUTER_RADIUS, 7/4*np.pi + CONNECTOR_ANGLE/2)
position_11 = myfunc.assemble_point(11, OUTER_RADIUS, 3/4*np.pi - CONNECTOR_ANGLE/2)
position_12 = myfunc.assemble_point(12, OUTER_RADIUS, 3/4*np.pi + CONNECTOR_ANGLE/2)

## Define Functions and Paramters

For the general linear differential equation 

\begin{equation}
-\frac{\partial}{\partial x}\left(\alpha_1(x, y) \frac{\partial \Phi(x, y)}{\partial x}\right)-\frac{\partial}{\partial y}\left(\alpha_2(x, y) \frac{\partial \Phi(x, y)}{\partial y}\right)+\beta(x, y) \Phi(x, y)=f(x, y)
\end{equation}

we substitue the generic functions with the given terms. Additionally, some parameters are set.

In [ ]:
def get_conductivity(x, y):

    radius = (x**2 + y**2)**0.5

    if radius <= INNER_RADIUS or abs(x) <= 0.5 * MATERIAL_WIDTH:
        if y > 0:
            return CONDUCTIVITY_3
        else:
            return CONDUCTIVITY_4
    elif x > 0.5 * MATERIAL_WIDTH:
        return CONDUCTIVITY_2
    elif x < -0.5 * MATERIAL_WIDTH:
        return CONDUCTIVITY_1
    else:
        raise Exception(f"Undefined conductivity at position {x}, {y}")
    
def alpha1(x, y):
    return 1 / get_conductivity(x, y)

def alpha2(x, y):
    return alpha1(x, y)

def beta(x, y):
    return 0

def rhs(x, y):
    return 0

In [ ]:
myfunc.plot_scalar_func(nodes, alpha1, labels=["x", "y", r"$\alpha_1(x,y)$"])

## Boundaray Condtions

### Dirichlet Boundary

For all points inside the area $G_D$ the solution value $\Phi$ is known.

\begin{equation}

    \Phi(x,y) = \delta(x,y) \quad x,y \in G_D

\end{equation}

### Robin Boundary

For all points inside the area $G_R$ the solution and its partial derivates are restricted.

\begin{equation}

    \left( \alpha_1(x,y) \frac{\partial \Phi(x, y)}{\partial x}, \alpha_2(x,y) \frac{\partial \Phi(x,y)}{\partial y} \right) \cdot \vec{n} + \gamma(x,y) \Phi(x,y) = \rho(x,y)

    \\ \quad

    x,y \in G_R

\end{equation}

In [ ]:
def dirichlet_func(x, y):
    if x >= position_11["x"] and y >= position_10["y"]:
        return 0
    else:
        return 1 / HEIGHT


def robin_gamma(x, y):
    return 0

def robin_rhs(x, y):
    return 0

In [ ]:
BOUNDARY_POSITIONS = []
for position in [position_9, position_10, position_11, position_12, position_9]:
    index = position["index"]
    x, y = position["x"], position["y"]

    BOUNDARY_POSITIONS += [[x, y]]

PATH_TYPES = ["Segments", "Nodes", "Segments", "Nodes"]
bseg = mt.RetrieveSegments(nodes,boundary_edge,boundary_indices, BOUNDARY_POSITIONS, PATH_TYPES)

DIRICHLET_INDICES = np.concat([bseg[1], bseg[3]])
ROBIN_INDICES = np.concat([bseg[0], bseg[2]])

In [ ]:
myfunc.plot_boundary(nodes, dirichlet_func, DIRICHLET_INDICES, ROBIN_INDICES)

## Solve 2D FEM

In [ ]:
stiffness_matrix, load_vector = myfunc.assemble_global_system(nodes, elements, alpha1, alpha2, beta, rhs)
stiffness_matrix, load_vector = myfunc.insert_robin_values(nodes, stiffness_matrix, load_vector, ROBIN_INDICES, robin_gamma, robin_rhs)
stiffness_matrix, load_vector = myfunc.insert_dirichlet_values(nodes, stiffness_matrix, load_vector, DIRICHLET_INDICES, dirichlet_func)

In [ ]:
solution = myfunc.solve_system(nodes, stiffness_matrix, load_vector)
myfunc.plot_result(elements, solution, levels=20, labels=["x", "y", r"$S(x,y)$"], title="Current Lines")

In [ ]:
solution["Phi"][3291]